In [4]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] ="2"

In [5]:
import torch
# 이후부터 torch는 물리 GPU 2만 “보임”
print(torch.cuda.device_count())  # 보통 1
print(torch.cuda.get_device_name(0))  # 물리 2번의 이름

1
NVIDIA GeForce RTX 3090


In [6]:
import os
import sys
import time
import json
import shutil
import subprocess
from datetime import datetime
from pathlib import Path
import pprint

# ====== 0) 반드시 프로젝트 루트 지정 ======
# configs.py /  / data/ / checkpoints/ / results/ 가 있는 폴더로!
PROJECT_ROOT = Path("/home/perseverance/lim/openstl-env/projects").resolve()

# (선택) 현재 커널의 cwd도 루트로 이동
os.chdir(PROJECT_ROOT)

# ====== 1) GPU 하나로 고정 (서브프로세스에도 적용) ======
GPU_ID = "2"   # 원하는 GPU 번호
BASE_ENV = os.environ.copy()
BASE_ENV["CUDA_VISIBLE_DEVICES"] = GPU_ID

# ====== 2) 로그 디렉토리 ======
LOG_DIR = PROJECT_ROOT / "logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)

# ====== 3) 공통 베이스 ======
BASE_CFG = {
    "data_path": "./data/oisst_ecs_2533_122130.zarr",
    "mask_path": "./data/oisst_spatial_mask_ecs.npy",
    "stats_cache": "./checkpoints/train_stats.json",
    "ckpt_dir": "./checkpoints",

    "var_name": "sst",
    "input_len": 14,
    "pred_len": 7,

    "train_start": "1983-01-01",
    "train_end": "2015-12-31",
    "val_start": "2016-01-01",
    "val_end": "2020-12-31",
    "test_start": "2021-01-01",
    "test_end": "2025-12-31",

    "lr": 1e-4,
    "weight_decay": 0.0,
    "batch_size": 64,
    "num_workers": 4,
    "max_epoch": 200,
    "patience": 20,
    "seed": 42,
    "device": "cuda",
}

# ====== 4) 모델별 추가 cfg ======
MODEL_CFGS = {
    "simvp_incepu": {
        "results_csv": "./results/simvp_incepu_results.csv",
        "exp_name": "simvp_incepu",
        "hid_S": 16,
        "hid_T": 64,
        "N_S": 4,
        "N_T": 4,
        "model_type": "incepu",
    },

    "convlstm": {
        "results_csv": "./results/convlstm_results.csv",
        "exp_name": "convlstm",
        "patch_size": 1,
        "in_shape": (14, 1, 32, 32),
        "filter_size": 3,
        "stride": 1,
        "layer_norm": False,
        "num_layers": 2,
        "num_hidden": [32, 32],
    },
    "predrnn": {
        "results_csv": "./results/predrnn_results.csv",
        "exp_name": "predrnn",
        "in_shape": (14, 1, 32, 32),
        "patch_size": 1,
        "num_layers": 2,
        "num_hidden": [32, 32],
        "filter_size": 3,
        "stride": 1,
        "layer_norm": 0,
        "reverse_scheduled_sampling": 0,
    },

    "mim": {
        "results_csv": "./results/MIM_results.csv",
        "exp_name": "MIM",
        "in_shape": (14, 1, 32, 32),
        "patch_size": 1,
        "num_layers": 2,
        "num_hidden": [32, 32],
        "filter_size": 3,
        "stride": 1,
        "layer_norm": 0,
    },
    "mau": {
        "results_csv": "./results/MAU_results.csv",
        "exp_name": "MAU",
        "tau": 5,
        "cell_mode": "normal",
        "model_mode": "normal",
        "sr_size": 1,
        "patch_size": 1,
        "filter_size": 3,
        "stride": 1,
        "layer_norm": 0,
        "num_hidden": [32, 32],
        "num_layers": 2,
    },

    "vit_gru": {
        "results_csv": "./results/vit_gru_results.csv",
        "exp_name": "vit_gru",
        "image_size": 32,
        "patch_size": 4,
        "embed_dim": 128,
        "depth": 3,
        "num_heads": 4,
        "mlp_ratio": 4.0,
        "vit_dropout": 0.0,
        "vit_attn_dropout": 0.0,
        "vit_freeze": False,
    },
    "vit": {
        "results_csv": "./results/vit_results.csv",
        "exp_name": "vit",
        "image_size": 32,
        "patch_size": 4,
        "embed_dim": 128,
        "depth": 3,
        "num_heads": 4,
        "mlp_ratio": 4.0,
        "vit_dropout": 0.0,
        "vit_attn_dropout": 0.0,
        "vit_freeze": False,
        "rollout_mode": "ar",
        "in_shape": (14, 1, 32, 32),
    },

    "timsformer": {
        "results_csv": "./results/TimeSformer_results.csv",
        "exp_name": "TimeSformer",
        "spatial_patch": 2,
        "d_model": 128,
        "depth": 3,
        "num_heads": 4,
        "mlp_ratio": 4.0,
        "dropout": 0.0,
    },

    "timsformer2": {
        "results_csv": "./results/TimeSformer2_results.csv",
        "exp_name": "TimeSformer2",
        "spatial_patch": 2,
        "d_model": 128,
        "depth": 3,
        "num_heads": 4,
        "mlp_ratio": 4.0,
        "dropout": 0.0,
    },
    "msf": {
        "results_csv": "./results/msf_results.csv",
        "exp_name": "msf",
        "d_model": 128,
        "depth": 3,
        "num_heads": 4,
        "mlp_ratio": 4.0,
        "dropout": 0.0,
        "attn_dropout": 0.0,
        "spatial_scales": (4, 8, 16, 32),
        "base_scale": 4,
        "use_pad": True,
        "use_pos_emb": False,
        "fusion": "concat",
    },
    "msfv2": {
        "results_csv": "./results/msfv2_results.csv",
        "exp_name": "msfv2",
        "d_model": 128,
        "depth": 3,
        "num_heads": 4,
        "mlp_ratio": 4.0,
        "dropout": 0.0,
        "attn_dropout": 0.0,
        "spatial_scales": (4, 8, 16),
        "time_pool": "last",
        "use_pad": True,
        "use_pos_emb": False,
        "fusion": "sum",
    },
    # 주의: swinlstm_train.py 하나로 D/B를 다룬다면 "exp_name"만 바꿔서 돌리는 게 맞음
    "swinlstm_D": {
        "results_csv": "./results/swinLstm_D_results.csv",
        "exp_name": "swinLstm_D",
        "depths_downsample": [2, 2, 2],
        "depths_upsample": [2, 2, 2],
        "num_heads": [2, 4, 8],
        "in_shape": (14, 1, 32, 32),
        "patch_size": 2,
        "embed_dim": 32,
        "window_size": 4,
        "depths": 3,
    },
    "swinlstm_B": {
        "results_csv": "./results/swinLstm_B_results.csv",
        "exp_name": "swinLstm_B",
        "in_shape": (14, 1, 32, 32),
        "num_heads": 4,
        "patch_size": 1,
        "embed_dim": 128,
        "window_size": 4,
        "depths": 3,
    },


    # MSFv2 MoM-Fusion (3 variants)

    "msfv2_gate": {
        "results_csv": "./results/msfv2_gate.csv",
        "exp_name": "msfv2_gate",

        "mom_top_k": None,          # None=soft mixture
        "mom_temperature": 1.0,

        "aux_balance_coef": 1e-2,
        "aux_z_coef": 1e-3,

        "aux_coef": 1.0,
        "drop_last": True,

        "d_model": 64,
        "depth": 3,
        "num_heads": 4,
        "mlp_ratio": 4.0,
        "dropout": 0.0,
        "attn_dropout": 0.0,
        "spatial_scales": (2, 4, 8, 16),
        "time_pool": "last",
        "use_pad": True,
        "use_pos_emb": False,
        "image_size": 32,
    },



    "msfv2_scalehead_moe_soft2": {
        "results_csv": "./results/msfv2_scalehead_moe_soft2.csv",
        "exp_name": "msfv2_scalehead_moe_soft2",

        "lambda_gdl": 0.1,
        "fusion": "sum",            # 처음엔 sum 추천 (효과 분리)
        "head_num_experts": 4,
        "head_top_k": None,         # soft mixture
        "head_temperature": 1.0,

        "aux_balance_coef": 1e-2,
        "aux_z_coef": 1e-3,

        "aux_coef": 1.0,
        "drop_last": True,

        "d_model": 64,
        "depth": 3,
        "num_heads": 4,
        "mlp_ratio": 4.0,
        "dropout": 0.0,
        "attn_dropout": 0.0,
        "spatial_scales": (2, 4, 8, 16),
        "time_pool": "last",
        "use_pad": True,
        "use_pos_emb": False,
        "image_size": 32,
    },

    "msfv2_scalehead_moe_top22": {
        "results_csv": "./results/msfv2_scalehead_moe_top22.csv",
        "exp_name": "msfv2_scalehead_moe_top22",

        "lambda_gdl": 0.02,
        "fusion": "sum",
        "head_num_experts": 4,
        "head_top_k": 2,            # top-2
        "head_temperature": 1.0,

        "aux_balance_coef": 1e-2,
        "aux_z_coef": 1e-3,

        "aux_coef": 1.0,
        "drop_last": True,

        "d_model": 64,
        "depth": 3,
        "num_heads": 4,
        "mlp_ratio": 4.0,
        "dropout": 0.0,
        "attn_dropout": 0.0,
        "spatial_scales": (2, 4, 8, 16),
        "time_pool": "last",
        "use_pad": True,
        "use_pos_emb": False,
        "image_size": 32,
    },

    "msfv2_scalehead_moe_top12": {
        "results_csv": "./results/msfv2_scalehead_moe_top12.csv",
        "exp_name": "msfv2_scalehead_moe_top12",

        "lambda_gdl": 0.02,
        "fusion": "sum",
        "head_num_experts": 4,
        "head_top_k": 1,            # top-1
        "head_temperature": 1.0,

        "aux_balance_coef": 1e-2,
        "aux_z_coef": 1e-3,

        "aux_coef": 1.0,
        "drop_last": True,

        "d_model": 64,
        "depth": 3,
        "num_heads": 4,
        "mlp_ratio": 4.0,
        "dropout": 0.0,
        "attn_dropout": 0.0,
        "spatial_scales": (2, 4, 8, 16),
        "time_pool": "last",
        "use_pad": True,
        "use_pos_emb": False,
        "image_size": 32,
    },
}

# ====== 5) 실행 커맨드 ======
RUN_CMDS = {
    # "simvp_incepu": ["uv", "run", "simvp_train.py"],
    # "convlstm":   ["uv", "run", "convlstm_train.py"],
    # "predrnn":    ["uv", "run", "predrnn_train.py"],
    # "mim":        ["uv", "run", "MIM_train.py"],
    # "mau":        ["uv", "run", "MAU_train.py"],
    # "vit_gru":    ["uv", "run", "vit_gru_train.py"],
    # "vit":        ["uv", "run", "vit_train.py"],
    # "timsformer": ["uv", "run", "timsformer_train.py"],
    # "timsformer2":["uv", "run", "timsformer_v2_train.py"],
    # "msf":        ["uv", "run", "msf_train.py"],
    # "msfv2":      ["uv", "run", "msf2_train.py"],

    # "msfv2_gate": ["uv", "run", "msf2_gate_train.py"],

    "msfv2_scalehead_moe_soft2": ["uv", "run", "msf2_scalehead_moe_train.py"],
    "msfv2_scalehead_moe_top22": ["uv", "run", "msf2_scalehead_moe_train.py"],
    "msfv2_scalehead_moe_top12": ["uv", "run", "msf2_scalehead_moe_train.py"],
    # "swinlstm_D": ["uv", "run", "swinlstm_train.py"],
    # "swinlstm_B": ["uv", "run", "swinlstm_train.py"],
}

RUN_PLAN = [
    # "simvp_incepu",
    # "convlstm",
    # "predrnn",
    # "mim",
    # "mau",
    # "vit_gru",
    # "vit",
    # "timsformer",
    # "timsformer2",
    # "msf",
    # "msfv2",
    # "swinlstm_D",
    # "swinlstm_B",
    # "msfv2_gate",

    "msfv2_scalehead_moe_soft2",
    "msfv2_scalehead_moe_top22",
    "msfv2_scalehead_moe_top12",
]

# ====== 6) configs.py를 "항상 올바른 파이썬"으로 덮어쓰기 ======
CONFIG_PATH = PROJECT_ROOT / "configs.py"
BACKUP_PATH = PROJECT_ROOT / "configs.py.bak"

def write_configs_py(cfg: dict):
    # 백업(1회)
    if CONFIG_PATH.exists() and not BACKUP_PATH.exists():
        shutil.copy2(CONFIG_PATH, BACKUP_PATH)

    body = pprint.pformat(cfg, width=120, sort_dicts=False)
    content = (
        "# AUTO-GENERATED by nightly runner\n"
        "def get_default_cfg():\n"
        "    return " + body + "\n"
    )
    tmp = CONFIG_PATH.with_suffix(".py.tmp")
    tmp.write_text(content, encoding="utf-8")
    tmp.replace(CONFIG_PATH)

def run_one(model_name: str):
    cfg = dict(BASE_CFG)
    cfg.update(MODEL_CFGS[model_name])

    # configs.py 덮어쓰기
    write_configs_py(cfg)

    # 실행
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    log_path = LOG_DIR / f"{ts}__{model_name}.log"

    cmd = RUN_CMDS[model_name]
    print(f"\n=== RUN {model_name} ===")
    print("CWD:", str(PROJECT_ROOT))
    print("CUDA_VISIBLE_DEVICES:", BASE_ENV.get("CUDA_VISIBLE_DEVICES"))
    print("CMD:", " ".join(cmd))
    print("LOG:", str(log_path))

    with open(log_path, "w", encoding="utf-8") as f:
        p = subprocess.run(
            cmd,
            cwd=str(PROJECT_ROOT),     
            env=BASE_ENV,              
            stdout=f,
            stderr=subprocess.STDOUT,
            text=True,
        )

    ok = (p.returncode == 0)
    print("OK" if ok else f"FAIL (returncode={p.returncode}) -> {log_path}")
    return {"model": model_name, "ok": ok, "returncode": p.returncode, "log": str(log_path)}

results = []
for m in RUN_PLAN:
    try:
        results.append(run_one(m))
    except Exception as e:
        ts = datetime.now().strftime("%Y%m%d_%H%M%S")
        log_path = LOG_DIR / f"{ts}__{m}__EXCEPTION.log"
        log_path.write_text(str(e), encoding="utf-8")
        print(f"EXCEPTION in {m} -> {log_path}")
        results.append({"model": m, "ok": False, "returncode": None, "log": str(log_path), "exception": str(e)})

# 요약 저장
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
summary_path = LOG_DIR / f"__SUMMARY_{ts}.json"
summary_path.write_text(json.dumps(results, indent=2, ensure_ascii=False), encoding="utf-8")
print("\n=== SUMMARY ===")
for r in results:
    print(r)
print("Summary saved to:", summary_path)



=== RUN msfv2_scalehead_moe_soft2 ===
CWD: /home/perseverance/lim/openstl-env/projects
CUDA_VISIBLE_DEVICES: 2
CMD: uv run msf2_scalehead_moe_train.py
LOG: /home/perseverance/lim/openstl-env/projects/logs/20260223_163537__msfv2_scalehead_moe_soft2.log
OK

=== RUN msfv2_scalehead_moe_top22 ===
CWD: /home/perseverance/lim/openstl-env/projects
CUDA_VISIBLE_DEVICES: 2
CMD: uv run msf2_scalehead_moe_train.py
LOG: /home/perseverance/lim/openstl-env/projects/logs/20260223_225635__msfv2_scalehead_moe_top22.log
OK

=== RUN msfv2_scalehead_moe_top12 ===
CWD: /home/perseverance/lim/openstl-env/projects
CUDA_VISIBLE_DEVICES: 2
CMD: uv run msf2_scalehead_moe_train.py
LOG: /home/perseverance/lim/openstl-env/projects/logs/20260224_014544__msfv2_scalehead_moe_top12.log
OK

=== SUMMARY ===
{'model': 'msfv2_scalehead_moe_soft2', 'ok': True, 'returncode': 0, 'log': '/home/perseverance/lim/openstl-env/projects/logs/20260223_163537__msfv2_scalehead_moe_soft2.log'}
{'model': 'msfv2_scalehead_moe_top22', 'o